In [25]:
import leafmap as lm
import geopandas as gpd

## Set-up the basemap

In [26]:
m = lm.Map( center = [38.65428167189044, -90.25320053100587],    
        zoom=11.5,
        height="925px",
        basemap = "CartoDB.DarkMatter",
        zoom_control=False,
        draw_control=False,
        scale_control=False,
        fullscreen_control=False,
        toolbar_control=False,
)
nbhd = "https://raw.githubusercontent.com/lweber89/geocrime-stl/main/src/geocrime_stl/data/stl_neighborhoods.geojson"

gdf = gpd.read_file(nbhd)
gdf.crs = "EPSG:4326"

line_only_style = {
    "stroke": True,
    "color": "#718096",
    "weight": 1.0,      
    "opacity": 0.8,      
    "fillOpacity": 0.0, 
}

# 4. Add the data to the map
m.add_gdf(
    gdf, 
    layer_name="Neighborhood Bndy", 
    style=line_only_style,
    hover_style=line_only_style,      # Explicitly disables any hover actions/style changes
    highlight_style={},
    info_mode = None
)

m

Map(center=[38.65428167189044, -90.25320053100587], controls=(AttributionControl(options=['position', 'prefix'…

In [ ]:
gradients = {
    "Person": {
        0.2: "#000044",
        0.4: "#0022ff",
        0.7: "#00d2ff",
        1.0: "#e0f7fa"
    },
    "Property": {
        0.2: "#220022",
        0.4: "#4a00e0",
        0.7: "#ff007f",
        1.0: "#ffebee"
    },
    "Society": {
        0.2: "#001a11",
        0.4: "#004d40",
        0.7: "#00ff66",
        1.0: "#e8f5e9"
    }
}

In [ ]:
# 1. Filter your data for the chosen category (e.g., Person)
filtered_crime = crime_df[crime_df['category'] == 'Person']

# 2. Add the heatmap layer on top of your existing map 'm'
m.add_heatmap(
    data=filtered_crime,
    lat="latitude",
    lon="longitude",
    radius=30,
    blur=12,
    gradient=person_gradient,  # Swap out for property_gradient or society_gradient
    name="Crime Hotspots"
)

In [ ]:
import pandas as pd
import leafmap
import ipywidgets as widgets
from IPython.display import display

# --- 1. PRE-PROCESS DATA (Do this once or in the cell above) ---
# Ensure your datetime column is parsed correctly
df['date_time'] = pd.to_datetime(df['date_time'])

# Create a clean "Year-Month" string column for the dropdown (e.g., "2024-05")
df['year_month'] = df['date_time'].dt.to_period('M').astype(str)

# Get sorted, unique Year-Month strings present in the actual data
valid_periods = sorted(df['year_month'].unique())
categories = sorted(df['off_type'].unique()) # ['Person', 'Property', 'Society']

# --- 2. DEFINE GRADIENTS ---
gradients = {
    "Person": {0.4: "#0022ff", 0.7: "#00d2ff", 1.0: "#e0f7fa"},
    "Property": {0.4: "#4a00e0", 0.7: "#ff007f", 1.0: "#ffebee"},
    "Society": {0.4: "#004d40", 0.7: "#00ff66", 1.0: "#e8f5e9"}
}

# --- 3. BUILD THE INTERACTIVE WIDGETS ---
period_dropdown = widgets.Dropdown(
    options=valid_periods,
    value=valid_periods[-1], # Default to the most recent month
    description='Time Period:',
)

type_dropdown = widgets.Dropdown(
    options=categories,
    value=categories[0],
    description='Offense Type:',
)

# --- 4. INITIALIZE BASEMAP AND NEIGHBORHOODS ---
m = leafmap.Map(basemap="CartoDB.DarkMatter")

subtle_gray_style = {
     "stroke": True,
     "color": "#718096",  
     "weight": 1.0,       
     "opacity": 0.6,      
     "fillOpacity": 0.0,  
}

# Add your static St. Louis neighborhood lines once to the background
m.add_gdf(
    gdf, 
    layer_name="St. Louis Neighborhoods", 
    style=subtle_gray_style,
    hover_style=subtle_gray_style,   
    highlight_style={},            
    info_mode=None,                
    zoom_to_layer=True
)

# --- 5. THE DYNAMIC UPDATE FUNCTION ---
def update_heatmap(change=None):
    chosen_period = period_dropdown.value
    chosen_type = type_dropdown.value
    layer_name = f"Hotspots: {chosen_type} ({chosen_period})"
    
    # CRITICAL: Find and wipe out ANY previous heatmap layers to stop stacking
    # We loop through backwards to safely remove layers without shifting index issues
    for layer in list(m.layers):
        if layer.name.startswith("Hotspots:"):
            m.remove_layer(layer)
            
    # Filter the data down to the strict user selection
    filtered_df = df[(df['year_month'] == chosen_period) & (df['off_type'] == chosen_type)]
    
    # If no data exists for a combination, skip adding a layer
    if filtered_df.empty:
        return
        
    # Add the fresh, single heatmap layer
    m.add_heatmap(
        data=filtered_df,
        lat="latitude",   # Adjust to match your exact parquet column name
        lon="longitude",  # Adjust to match your exact parquet column name
        radius=30,
        blur=12,
        gradient=gradients[chosen_type],
        name=layer_name
    )

# Tie the dropdown changes directly to our update function
period_dropdown.observe(update_heatmap, names='value')
type_dropdown.observe(update_heatmap, names='value')

# Trigger the initial map draw for the default dropdown values
update_heatmap()

# --- 6. DISPLAY DASHBOARD ---
# Pack the controls into a neat horizontal layout above the map
controls = widgets.HBox([period_dropdown, type_dropdown])
display(controls)
display(m)